### 1. What is a Window Function?

#### A Window Function performs calculations across a group of related rows without collapsing those rows into a single result.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

data = [
    (1, 101, 250, "2026-01-01"),
    (2, 101, 300, "2026-01-05"),
    (3, 102, 150, "2026-01-02"),
    (4, 102, 400, "2026-01-06"),
    (5, 103, 200, "2026-01-03"),
    (6, 101, 120, "2026-01-10"),
    (7, 104, 500, "2026-01-04"),
    (8, 102, 350, "2026-01-08"),
    (9, 103, 180, "2026-01-07"),
    (10, 104, 220, "2026-01-09")
]

columns = ["order_id", "customer_id", "amount", "order_date"]

df = spark.createDataFrame(data, schema=columns)

####Using groupBy()

In [0]:
df.groupBy("customer_id").agg(
    F.sum("amount").alias("total_amount")
).show()

####Using Window Functions

In [0]:
window_spec = Window.partitionBy("customer_id")

df.withColumn(
    "total_amount",
    F.sum("amount").over(window_spec)
).display()

####partitionBy()
Divides data into logical groups

In [0]:
Window.partitionBy("customer_id").orderBy(
    F.col("order_id").desc()
)

### 3. row_number()

In [0]:
window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy("order_id")
)

result = df.withColumn(
    "row_num",
    F.row_number().over(window_spec)
)

result.display()

In [0]:
window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("order_date").desc())
)

result = (
    df.withColumn(
        "row_num",
        F.row_number().over(window_spec)
    ).filter(F.col("row_num") == 1)
    .drop("row_num")
)

df.orderBy(F.col("order_date").desc()).display()
result.display()

### 4. rank() and dense_rank()

In [0]:
data = [
    (101, "Ali", 500),
    (102, "John", 400),
    (103, "Sarah", 400),
    (104, "David", 300)
]

df = spark.createDataFrame(
    data,
    ["customer_id", "name", "total_sales"]
)

### rank()

In [0]:
window_spec = Window.orderBy(
    F.col("total_sales").desc()
)

result = df.withColumn(
    "ranking",
    F.rank().over(window_spec)
)

result.display()

### dense_rank()

In [0]:
result = df.withColumn(
    "ranking",
    F.dense_rank().over(window_spec)
)

result.display()

### 5. lag() and lead()

In [0]:
transactions = [
    (101, "2025-01-01", 100),
    (101, "2025-01-02", 150),
    (101, "2025-01-03", 200),
    (102, "2025-01-01", 300),
    (102, "2025-01-02", 400)
]

df = spark.createDataFrame(
    transactions,
    ["customer_id", "transaction_date", "amount"]
)

### lag()

In [0]:
window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy("transaction_date")
)

result = df.withColumn(
    "previous_amount",
    F.lag("amount", 1, 0).over(window_spec)
)

result.display()

### lead()

In [0]:
result = df.withColumn(
    "next_amount",
    F.lead("amount", 1).over(window_spec)
)

result.display()

### 6. Running Totals

In [0]:
window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy("order_id")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

result = df.withColumn(
    "running_total",
    F.sum("amount").over(window_spec)
)

result.display()

### 7. Deduplication using Window Functions

In [0]:
customer_data = [
    (101, "Ali", "2025-01-01"),
    (101, "Ali Ahmed", "2025-01-05"),
    (102, "John", "2025-01-02"),
    (102, "John Smith", "2025-01-06")
]

customer_columns = ["customer_id", "name", "updated_at"]

customer_df = spark.createDataFrame(customer_data, schema=customer_columns)

window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("updated_at").desc())
)

latest_customers = (
    customer_df.withColumn(
        "row_num",
        F.row_number().over(window_spec)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

latest_customers.display()

### 8. How Window Functions connect to CDC and SCD

In [0]:
audit_data = [
    (101, "Ali", "INSERT", "10:00"),
    (101, "Ali Ahmed", "UPDATE", "11:00"),
    (101, "Ali Khan", "UPDATE", "12:00"),
    (102, "John", "INSERT", "10:30")
]

audit_columns = ["customer_id", "name", "operation", "event_time"]

audit_df = spark.createDataFrame(audit_data, audit_columns)

window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("event_time").desc())
)

latest_changes = (
    audit_df
    .withColumn(
        "row_num",
        F.row_number().over(window_spec)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

latest_changes.display()